<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Assess distortions induced by subsidy/policy package variants.
- **Primary inputs**: runs/subsidies_distortions/*/output.csv and related policy indicator tables
- **Primary outputs**: distortion comparison tables and figures
- **Refactor role**: Keep as dedicated policy-assessment notebook; reuse shared run loading and naming conventions.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Load subsidy-distortion and scenario output tables for one run folder.
2. Create comparison tables by scenario/policy category.
3. Export year-specific subsidy comparison outputs and figures.

### Inputs
- `runs/subsidies_distortions/<run_id>/*/subsidies_distortion.csv`
- `runs/subsidies_distortions/<run_id>/*/output.csv`

### Outputs
- `runs/subsidies_distortions/<run_id>/img/` figures
- `runs/subsidies_distortions/<run_id>/img/subsidies_<year>.csv`

### How To Reuse
- Set `folder` (run id) and `select` year in code cells to target another analysis slice.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
folder = 'test'
# Refactor: subsidy-distortion runs are under policy_assessment/runs/subsidies_distortions
folder = os.path.join('runs', 'subsidies_distortions', folder)
folder_out = os.path.join(folder, 'img')
if not os.path.exists(folder_out):
    os.makedirs(folder_out)

In [ ]:
# in folder open all child folders and read the output.csv files
dict_output = {child_folder: pd.read_csv(os.path.join(folder, child_folder, 'subsidies_distortion.csv'), header=[0], index_col=None)
             for child_folder in os.listdir(folder)
             if os.path.isdir(os.path.join(folder, child_folder))
             and os.path.isfile(os.path.join(folder, child_folder, 'subsidies_distortion.csv'))}


In [ ]:
if False:
    sub = ['subsidies_2018', 'subsidies_2021', 'subsidies_2024', 'cee_2018', 'cee_2021', 'cee_2024']
    dict_output = {key: value for key, value in dict_output.items() if key in sub}
    # order the keys
    dict_output = {key: dict_output[key] for key in sub if key in dict_output.keys()}


In [ ]:
for k in dict_output.keys():
    dict_output[k]['Subsidies Gap'] = dict_output[k]['Distortion'] - dict_output[k]['Subsidies']

In [ ]:
# in folder open all child folders and read the output.csv files
dict_output = {child_folder: pd.read_csv(os.path.join(folder, child_folder, 'output.csv'), header=[0], index_col=[0])
             for child_folder in os.listdir(folder)
             if os.path.isdir(os.path.join(folder, child_folder))
             and os.path.isfile(os.path.join(folder, child_folder, 'output.csv'))}


In [ ]:
dict_output.keys()

In [ ]:
import re
select = 2019
select = 'avg'
result = {}

# Define the pattern you are looking for
pattern = r'^Subsidies total (.*) \(Million euro\)$'
ref =  r'^Investment total (.*) \(Billion euro\)$'
ref = None

for key, df in dict_output.items():
    # Filter rows whose index matches the pattern
    mask = df.index.to_series().str.match(pattern)
    filtered = df[mask].copy()

    # Update the index to be only the part inside the { ... }
    new_index = df.index[mask].to_series().str.extract(pattern)[0]
    filtered.index = new_index
    if ref is not None:
        mask = df.index.to_series().str.match(ref)
        filtered_ref = df[mask].copy()

        # Update the index to be only the part inside the { ... }
        new_index = df.index[mask].to_series().str.extract(ref)[0]
        filtered_ref.index = new_index

        filtered = filtered / (filtered_ref * 1e3)
        filtered.dropna(axis=0, how='all', inplace=True)

    # Collect the column values
    if select == 'avg':
        result[key] = filtered.mean(axis=1)
    else:
        result[key] = filtered[str(select)]

# Create final DataFrame
final_df = pd.DataFrame(result)
final_df.to_csv(os.path.join(folder_out, f'subsidies_{select}.csv'), index=True)

In [ ]:
same_ylim = True
fontsize = 15

axes = final_df.T.plot.bar(
    rot=0,
    subplots=True,
    layout=(2,3),
    figsize=(16, 12),  # Make figure bigger
    legend=False       # Remove legends
)

# Flatten axes if they come nested
if isinstance(axes, np.ndarray):
    axes = axes.flatten()
else:
    axes = [axes]


# Set same y-axis if requested
if same_ylim:
    # Find global min and max
    all_values = final_df.values.flatten()
    ymin, ymax = all_values.min(), all_values.max()

    # Add some margin
    margin = (ymax - ymin) * 0.05
    ymin -= margin
    ymax += margin

    for ax in axes:
        ax.set_ylim(ymin, ymax)

# Update fontsize for all elements
for ax in axes:
    ax.title.set_fontsize(fontsize)
    ax.xaxis.label.set_fontsize(fontsize)
    ax.yaxis.label.set_fontsize(fontsize)
    ax.tick_params(axis='both', labelsize=fontsize)

plt.tight_layout()
plt.show()




In [ ]:
final_df

In [ ]:
'Subsidies total {} (Million euro)'

In [ ]:
data = pd.concat(dict_output)
data['Scenario'] = data.index.get_level_values(0)

x = 'Scenario'
y = 'Subsidies Gap'
# plot boxplot of 'Subsidies Gap' for each key
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(12.8, 9.6))
# annotate the plot
sns.set_theme(style="whitegrid")
box_plot = sns.boxplot(data=data, x=x, y=y, ax=ax)

medians = data.groupby([x])[y].median()
vertical_offset = data[y].median() * 0.05 # offset from median for display

for xtick in box_plot.get_xticks():
    # format the median value with 2 decimal places

    box_plot.text(xtick,medians[xtick] + vertical_offset, f'{medians[xtick]:.0f}',
            horizontalalignment='center',size='x-small',color='w',weight='semibold')

plt.savefig(os.path.join(folder_out, 'subsidies_gap.png'))

In [ ]:
from project.utils import make_scatter_plot
from project.input.resources import resources_data

if True:
    xmin = min([dict_output[k]['Subsidies'].min() for k in dict_output.keys()])
    xmax = min([dict_output[k]['Subsidies'].max() for k in dict_output.keys()])
    ymin = min([dict_output[k]['Distortion'].min() for k in dict_output.keys()])
    ymax = min([dict_output[k]['Distortion'].max() for k in dict_output.keys()])
    
remove = ['Electricity-Direct electric',
 'Natural gas-Performance boiler',
 'Oil fuel-Performance boiler',
 'Wood fuel-Performance boiler',
 'Heating-District heating']

for key, temp in dict_output.items():
    temp = temp.reset_index()
    temp = temp[~temp['Technology'].isin(remove)]
    #temp = temp.groupby(['Occupancy status', 'Housing type', 'Performance', 'Energy', 'Income owner']).mean()
    col_colors = 'Income owner'
    # temp[col_colors] = temp[col_colors].apply(lambda x: resources_data['colors'][x])
    col_colors = None
    
    make_scatter_plot(temp, 'Subsidies', 'Distortion', 'Subsidies (Thousand euro)',
                      'Distortion (Thousand euro)',
                      annotate=False,
                      save=os.path.join(folder_out, 'subsidies_distortion_{}.png'.format(key)),
                      format_y=lambda y, _: '{:.0f}'.format(y/1e3),
                      format_x=lambda x, _: '{:.0f}'.format(x/1e3),
                      s=10, diagonal_line=True, col_colors=col_colors,
                      xmin=xmin, ymin=ymin, xmax=xmax, ymax=ymax)


In [ ]:
temp

In [ ]:
temp